In [5]:
# Escanea el mercado y trae los tickers cuyo precio este N% de su média móvil ponderada de 150 y 200 en SEMANAL.
# Correr sábado o domingo: todas las barras son semanas cerradas

In [6]:
#Importa las librerias y módulos necesarios.
%load_ext autoreload
%autoreload 2
import pandas as pd
from datos import obtener_tickers

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# Descarga de datos semanales.

data_7d = obtener_tickers("1wk", "max")

Se han cargado 538 tickers...


$USPTLT: possibly delisted; no timezone found

1 Failed download:
['USPTLT']: possibly delisted; no timezone found


Tiempo de descarga en 328.59 segundos.
Se descargó datos en temporalidad 1wk con un periodo de max


In [8]:
resultados = []

for ticker in data_7d.columns.get_level_values(0).unique():
    df_ticker = data_7d[ticker].dropna(subset=['Close', 'High', 'Low'])
    # Descartar la vela semanal en curso
    df_ticker = df_ticker.iloc[:-1]
    
    if len(df_ticker) >= 600:
        cierre = df_ticker['Close']
        high = df_ticker['High']
        
        s_ema_150 = cierre.ewm(span=150, adjust=False).mean()
        s_ema_200 = cierre.ewm(span=200, adjust=False).mean()
        
        ema_150 = s_ema_150.iloc[-1].item()
        ema_200 = s_ema_200.iloc[-1].item()
        precio = cierre.iloc[-1].item()

        # Condición 1: Estructura alcista y pendiente positiva en ambas medias
        pendiente_150 = ema_150 > s_ema_150.iloc[-14].item()
        pendiente_200 = ema_200 > s_ema_200.iloc[-14].item()
        estructura_valida = (ema_150 > ema_200) and pendiente_150 and pendiente_200

        if estructura_valida:
            dist_150 = ((precio - ema_150) / ema_150) * 100
            dist_200 = ((precio - ema_200) / ema_200) * 100

            # Condición 2: El precio está tocando la zona de la EMA (a 1% o menos)
            toque_150 = abs(dist_150) <= 1.0
            toque_200 = abs(dist_200) <= 1.0

            # Condición 3 (Antirrebote de techo): Validar que el toque sea DESDE ARRIBA.
            # Verificamos que en las últimas 8 semanas el precio haya estado operando
            # claramente por encima de la zona, garantizando que viene corrigiendo.
            max_previo_150 = high.iloc[-8:-1].max() > (ema_150 * 1.03)
            max_previo_200 = high.iloc[-8:-1].max() > (ema_200 * 1.03)

            soportes_validos = (toque_150 and max_previo_150) or (toque_200 and max_previo_200)

            if soportes_validos:
                resultados.append({
                    'ticker': ticker,
                    'precio': precio,
                    'dist_150': dist_150,
                    'dist_200': dist_200,
                })

for r in resultados:
    print(
        f"SOPORTE | Ticker: {r['ticker']} | Precio: {r['precio']:.2f} | "
        f"Dist. EMA 150: {r['dist_150']:.2f}% | Dist. EMA 200: {r['dist_200']:.2f}%"
    )

SOPORTE | Ticker: PAYX | Precio: 112.12 | Dist. EMA 150: 0.62% | Dist. EMA 200: 2.24%
SOPORTE | Ticker: GPC | Precio: 120.90 | Dist. EMA 150: -1.15% | Dist. EMA 200: -0.43%
SOPORTE | Ticker: CRM | Precio: 229.17 | Dist. EMA 150: 0.04% | Dist. EMA 200: 0.69%
SOPORTE | Ticker: VMRK | Precio: 61.90 | Dist. EMA 150: 0.27% | Dist. EMA 200: 1.30%
SOPORTE | Ticker: PCAR | Precio: 101.91 | Dist. EMA 150: -0.23% | Dist. EMA 200: 5.97%
SOPORTE | Ticker: TPL | Precio: 331.68 | Dist. EMA 150: -0.69% | Dist. EMA 200: 6.96%
SOPORTE | Ticker: T | Precio: 21.41 | Dist. EMA 150: -3.50% | Dist. EMA 200: 0.99%
SOPORTE | Ticker: UHS | Precio: 176.05 | Dist. EMA 150: -0.24% | Dist. EMA 200: 2.54%
SOPORTE | Ticker: BG | Precio: 96.64 | Dist. EMA 150: -0.61% | Dist. EMA 200: 3.19%
SOPORTE | Ticker: SLV | Precio: 44.60 | Dist. EMA 150: -0.76% | Dist. EMA 200: 8.83%
SOPORTE | Ticker: ODFL | Precio: 172.56 | Dist. EMA 150: -4.07% | Dist. EMA 200: -0.62%
SOPORTE | Ticker: CBRE | Precio: 128.85 | Dist. EMA 150: -